In [1]:
import sys
import os
from pathlib import Path

# 1. Ana qovluğu tapırıq
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# 2. İşçi qovluğu ANA qovluğa dəyişirik
os.chdir(PROJECT_ROOT)

# 3. Sənin strukturuna uyğun olaraq qovluğu ÖNCƏDƏN yaradırıq
# pipeline.py-nin axtardığı "logs/" qovluğunu src/logs olaraq yaradırıq
log_dir = PROJECT_ROOT / "src" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

# 4. 'src' qovluğunu Python yoluna əlavə edirik
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

# 5. İndi importları yoxlayaq
try:
    # pipeline.py-dəki xətanın qarşısını almaq üçün müvəqqəti olaraq 
    # həmin faylın axtardığı nisbi yolda boş bir log faylı yaradırıq (əgər yoxdursa)
    # (Bu addım pipeline.py-ni dəyişmədən xətanı keçmək üçündür)
    (PROJECT_ROOT / "logs").mkdir(exist_ok=True) 
    
    from database import get_connection
    from pipeline import run_full_pipeline, run_incremental_pipeline
    from logging_utils import setup_logger
    
    print(f"✅ Project Root: {PROJECT_ROOT}")
    print(f"✅ Logs Directory: {log_dir}")
    print("✅ Imports successful")
except Exception as e:
    print(f"❌ Import xətası: {e}")

✅ Project Root: C:\Users\user\Documents\IronHack_Labs\weather-intelligence-pipeline-team
✅ Logs Directory: C:\Users\user\Documents\IronHack_Labs\weather-intelligence-pipeline-team\src\logs
✅ Imports successful


In [2]:
# 1. Qovluqları düzgün yerlərdə yaradırıq
(PROJECT_ROOT / "src" / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data" / "database").mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data" / "raw").mkdir(parents=True, exist_ok=True)

# 2. Database yolunu təyin edirik (data/database/ içində)
db_path = PROJECT_ROOT / "data" / "database" / "weather_daily.duckdb"

# 3. Bağlantı qururuq
conn = get_connection(str(db_path))

print(f"✅ Database Path: {db_path}")
print(f"✅ Logs Directory: {PROJECT_ROOT / 'src' / 'logs'}")

✅ Database Path: C:\Users\user\Documents\IronHack_Labs\weather-intelligence-pipeline-team\data\database\weather_daily.duckdb
✅ Logs Directory: C:\Users\user\Documents\IronHack_Labs\weather-intelligence-pipeline-team\src\logs


In [3]:
# Bazada cədvəllərin olub-olmadığını yoxlayırıq
try:
    tables = conn.execute("SHOW TABLES").fetchall()
    print("Mövcud cədvəllər:", tables)
except Exception as e:
    print("Cədvəllər hələ yaradılmayıb (Full Pipeline-dan sonra yaranacaq).")

Mövcud cədvəllər: []


In [4]:
# Logger-i işə salırıq
logger = setup_logger()

logger.info("===== FULL PIPELINE START =====")
# Full pipeline işləyir
df_full = run_full_pipeline(conn, logger)
logger.info("===== FULL PIPELINE END =====")

print("✅ Full pipeline completed successfully!")
display(df_full)

Fetching → Baku
Baku: 1827 rows
Fetching → Ganja
Ganja: 1827 rows
Fetching → Shusha
Shusha: 1827 rows
Fetching → Nakhchivan
Nakhchivan: 1827 rows
Loading Historical CSV: Baku_historical.csv
Loading Historical CSV: Ganja_historical.csv
Loading Historical CSV: Nakhchivan_historical.csv
Loading Historical CSV: Shusha_historical.csv
Loading Forecast CSV: Baku_forecast.csv
Loading Forecast CSV: Ganja_forecast.csv
Loading Forecast CSV: Nakhchivan_forecast.csv
Loading Forecast CSV: Shusha_forecast.csv

══════════════════════════════════════════════════
  Processing raw.weather_daily_historical …
  Rows loaded : 7,308
  Handling missing values …
  Nulls before: 0  →  after: 0
  Flagging outliers (IQR, threshold=1.5) …
  Total outlier flags: 4,155  across 12 columns
  Validating date continuity …

─────────────────────────────────────────────
  City            : Baku
  Date range      : 2021-04-17 → 2026-04-17
  Expected days   : 1827
  Actual days     : 1827
  Missing days    : 0
─────────────

{'historical_rows': 7308,
 'forecast_rows': 28,
 'staging_hist_rows': 7308,
 'analytics_hist_rows': 7308,
 'analytics_forecast_rows': 28,
 'duration': '0:00:07.417895'}

In [5]:
hist_count = conn.execute("SELECT COUNT(*) FROM raw.weather_daily_historical").fetchone()[0]
feat_count = conn.execute("SELECT COUNT(*) FROM analytics.weather_features_historical").fetchone()[0]

print(f" RAW HIST ROWS: {hist_count}")
print(f" ANALYTICS FEATURES ROWS: {feat_count}")

 RAW HIST ROWS: 7308
 ANALYTICS FEATURES ROWS: 7308


In [6]:
logger.info("===== INCREMENTAL PIPELINE START =====")
df_inc = run_incremental_pipeline(conn, logger)
logger.info("===== INCREMENTAL PIPELINE END =====")

print("✅ Incremental run completed.")
display(df_inc)

Fetching → Baku


RuntimeError: API failed after 3 retries: failed to request 'https://archive-api.open-meteo.com/v1/archive': {'reason': 'Minutely API request limit exceeded. Please try again in one minute.', 'error': True}

In [7]:
print("Running incremental again...")

df_inc_2 = run_incremental_pipeline(conn, logger)

df_inc_2

Running incremental again...

══════════════════════════════════════════════════
  Processing raw.weather_daily_historical …
  Rows loaded : 7,328
  Handling missing values …
  Nulls before: 0  →  after: 0
  Flagging outliers (IQR, threshold=1.5) …
  Total outlier flags: 4,178  across 12 columns
  Validating date continuity …

─────────────────────────────────────────────
  City            : Baku
  Date range      : 2021-04-17 → 2026-04-21
  Expected days   : 1831
  Actual days     : 1831
  Missing days    : 0
─────────────────────────────────────────────

─────────────────────────────────────────────
  City            : Ganja
  Date range      : 2021-04-17 → 2026-04-21
  Expected days   : 1831
  Actual days     : 1831
  Missing days    : 0
─────────────────────────────────────────────

─────────────────────────────────────────────
  City            : Nakhchivan
  Date range      : 2021-04-17 → 2026-04-21
  Expected days   : 1831
  Actual days     : 1831
  Missing days    : 0
─────────

{'mode': 'incremental',
 'rows_added_raw': 0,
 'affected_cities': [],
 'skipped_cities': ['Baku', 'Ganja', 'Shusha', 'Nakhchivan'],
 'raw_hist_total': 7328,
 'raw_forecast_total': 28,
 'staging_hist': 7328,
 'staging_forecast': 28,
 'analytics_hist': 7328,
 'analytics_forecast': 28,
 'timestamp': '2026-04-22 21:18:02.122569'}

In [8]:
print("RAW HIST:",
      conn.execute("SELECT COUNT(*) FROM raw.weather_daily_historical").fetchone())

print("RAW FORECAST:",
      conn.execute("SELECT COUNT(*) FROM raw.weather_daily_forecast").fetchone())

RAW HIST: (7328,)
RAW FORECAST: (28,)


In [9]:
log_path = PROJECT_ROOT / "logs" / "pipeline.log"

if log_path.exists():
    with open(log_path, "r", encoding="utf-8") as f:
        logs = f.readlines()
    print(f"✅ Log file found at: {log_path}")
    print("\nSon 15 log qeydi:")
    print("".join(logs[-15:]))
else:
    print(f"❌ XƏTA: Log faylı bu yolda tapılmadı: {log_path}")

✅ Log file found at: C:\Users\user\Documents\IronHack_Labs\weather-intelligence-pipeline-team\logs\pipeline.log

Son 15 log qeydi:
2026-04-22 21:18:00,455 | weather_pipeline | INFO | Ganja: +5 rows
2026-04-22 21:18:00,547 | weather_pipeline | INFO | Shusha: +5 rows
2026-04-22 21:18:00,642 | weather_pipeline | INFO | Nakhchivan: +5 rows
2026-04-22 21:18:00,971 | weather_pipeline | INFO |  Rebuilding staging tables
2026-04-22 21:18:01,054 | weather_pipeline | INFO |  Feature engineering started
2026-04-22 21:18:01,087 | weather_pipeline | INFO |  INCREMENTAL SUMMARY: {'mode': 'incremental', 'rows_added_raw': 20, 'affected_cities': ['Baku', 'Ganja', 'Shusha', 'Nakhchivan'], 'skipped_cities': [], 'raw_hist_total': 7328, 'raw_forecast_total': 28, 'staging_hist': 7328, 'staging_forecast': 28, 'analytics_hist': 7328, 'analytics_forecast': 28, 'timestamp': '2026-04-22 21:18:01.087065'}
2026-04-22 21:18:01,087 | weather_pipeline | INFO | ===== INCREMENTAL PIPELINE END =====
2026-04-22 21:18:01,

In [10]:
print("""
================ PIPELINE ARCHITECTURE ================

        WEATHER API
             │
             ▼
     INGESTION LAYER
             │
             ▼
       RAW (DuckDB)
             │
             ▼
     STAGING (CLEANING)
             │
             ▼
   FEATURE ENGINEERING
             │
             ▼
     ANALYTICS LAYER
             │
             ▼
     QUALITY CHECKS
             │
             ▼
        LOGGING SYSTEM

FULL MODE:
- Full rebuild of dataset

INCREMENTAL MODE:
- Only new data ingestion
- Skips if up-to-date

======================================================
""")


================ PIPELINE ARCHITECTURE ================

        WEATHER API
             │
             ▼
     INGESTION LAYER
             │
             ▼
       RAW (DuckDB)
             │
             ▼
     STAGING (CLEANING)
             │
             ▼
   FEATURE ENGINEERING
             │
             ▼
     ANALYTICS LAYER
             │
             ▼
     QUALITY CHECKS
             │
             ▼
        LOGGING SYSTEM

FULL MODE:
- Full rebuild of dataset

INCREMENTAL MODE:
- Only new data ingestion
- Skips if up-to-date




In [11]:
print("""
RESULT SUMMARY:

✅ Full pipeline executed successfully
✅ Data ingested → cleaned → transformed
✅ Features created
✅ Quality checks executed
✅ Logs written to pipeline.log
✅ Incremental mode handled correctly

TASK 5 COMPLETED SUCCESSFULLY 🚀
""")


RESULT SUMMARY:

✅ Full pipeline executed successfully
✅ Data ingested → cleaned → transformed
✅ Features created
✅ Quality checks executed
✅ Logs written to pipeline.log
✅ Incremental mode handled correctly

TASK 5 COMPLETED SUCCESSFULLY 🚀

